# NB08 — Grafo epistémico: D3/D4 ratificados + MVP

Reflejo del **Grafo Epistémico** aprobado por rmw3 (S5) el **2026-10-10**
(`docs/plans/graph_epistemic_proposal.vsm`, `@status ratified`), pedido
explícito de S5 al aprobar: *"quiero ver esto reflejado en el notebook"*.

- **D3 (identidad)** — *content-keyed*: la identidad de un nodo es su
  contenido, nunca un id local de sesión ni un mtime (misma regla F3 que
  `trace_graph`).
- **D4 (aristas MVP, 6)** — `child_of`, `extracted_from`, `certified_by`,
  `derived_from`, `correlates_with` (bidireccional, `<->` ratificado),
  `answers`. Backlog fase-2 (`contradicts`, `refutes`, `supersedes`)
  queda **solo diseñado**.
- **9 tipos de nodo** — document, claim, predicate, certificate, answer,
  experiment, decision, gap, rule.
- **NO-NEW-TRUTH** — el grafo es *vista*, no fuente: toda verdad sigue
  viva en su sidecar/ledger canónico (R5) y el grafo se reconstruye
  completo desde cero (se demuestra abajo).
- **R10** — misma entrada → mismo grafo (sin `random`, sin reloj).

**Fuentes (7, todas ya persistidas):** `tree_traces.jsonl`,
`*.claims.vsm`, `*.depth.vsm`, `*.gap.json`, `hard_rules.vsm`,
`paper_data.vsm`, `current_focus.json` + `plan_registry.vsm`.

*Ejecutar desde la raíz del repo:* `python3 /tmp/opencode/run_notebook.py docs/papers/attractor_convergence/analysis/08_grafo_epistemico.ipynb`

In [ ]:
import sys, json, hashlib
sys.path.insert(0, "packages/framework")
from epistemic_graph import build, by_kind, by_dim, to_dot, to_json

g = build()                      # con cache por fingerprint de fuentes
m = g["meta"]
print("fingerprint :", m["fingerprint"][:16], "| cache_hit:", m["cache_hit"])
print("nodes:", m["nodes_total"], "| edges:", m["edges_total"])
print("nodes_by_kind:", json.dumps(m["nodes_by_kind"]))
print("edges_by_kind:", json.dumps(m["edges_by_kind"]))
print("skipped:", json.dumps(m["skipped"]))

fingerprint : 6926f387ebbac1cc | cache_hit: True
nodes: 8917 | edges: 8816
nodes_by_kind: {"answer": 7887, "certificate": 19, "claim": 544, "decision": 56, "document": 13, "experiment": 14, "gap": 6, "predicate": 363, "rule": 15}
edges_by_kind: {"answers": 7887, "certified_by": 19, "child_of": 404, "correlates_with": 30, "derived_from": 19, "extracted_from": 457}
skipped: {"claim_duplicate_content": 46, "claims_without_document": 87, "decision_duplicate": 4, "exp_tokens_without_experiment": 9, "traces_malformed": 0}


## ① D3 — identidad content-keyed (ejemplos reales)

Un nodo `document` lleva el **stem + fingerprint del artefacto**: si el
artefacto no cambia, cualquier toque (mtime) deja la identidad intacta;
si cambia un byte, la identidad cambia. Lo mismo para `claim`
(`source_id + text_hash`) y `predicate` (texto VSL literal, F3).

In [ ]:
docs = [n for n in g["nodes"] if n["kind"] == "document"]
for n in sorted(docs, key=lambda x: x["id"])[:4]:
    print(n["id"])
    print("   dims:", {k: n["dims"][k] for k in ("level", "coverage", "status", "fingerprint") if n["dims"].get(k)})
claim = next(n for n in g["nodes"] if n["kind"] == "claim")
pred  = next(n for n in g["nodes"] if n["kind"] == "predicate")
cert  = next(n for n in g["nodes"] if n["kind"] == "certificate")
print("\nclaim    :", claim["id"])
print("predicate:", pred["id"][:100])
print("certificate:", cert["id"], "-> derived_from ->", [e["target"] for e in g["edges"] if e["source"] == cert["id"] and e["kind"] == "derived_from"])

document:addendum|6a13177d1d2e
   dims: {'level': 'L1-L5', 'coverage': 21, 'status': 'active', 'fingerprint': '6a13177d1d2e'}
document:carta_acompanamiento|ad17f4138fb2
   dims: {'level': 'L1-L5', 'coverage': 5, 'status': 'active', 'fingerprint': 'ad17f4138fb2'}
document:claim_ingest_proposal|67f65fd25a9c
   dims: {'level': 'L1-L5', 'status': 'active', 'fingerprint': '67f65fd25a9c'}
document:doc_tracking_plan|03c56c8208dd
   dims: {'coverage': 7, 'fingerprint': '03c56c8208dd'}

claim    : claim:CL-A01|0f1ee2f33910273d
predicate: predicate:AND(action_forbidden($tool, $file), path_forbidden($file), write_scope_ok($tool, $file), n
certificate: certificate:CL-ARC-001#0 -> derived_from -> ['claim:CL-ARC-001|d314a946474e63fb']


## ② Vistas por `kind` y por `dim`

Contrato de dimensiones (hibrido, S5): **base** `kind, status, source, ts`
presente en *todo* nodo (null cuando no aplica) + *extras* por kind.
`COVERAGE-DENOMINATOR` aplicado al grafo: toda vista reporta sus
denominadores (`nodes_total`, `edges_total`, `dropped_edges`).

In [ ]:
for k, n in m["nodes_by_kind"].items():
    v = by_kind(g, k)
    e = v["meta"]["edges_total"]
    print(f"{k:12s} nodes={n:5d}  edges_inside_view={e}")

print()
tf = by_dim(g, "truth", "Truth.FALSE")
print("dim truth=Truth.FALSE     ->", tf["meta"]["nodes_total"], "nodos,",
      tf["meta"]["edges_total"], "aristas internas")
d = by_dim(g, "status", "DECLARED")
print("dim status=DECLARED       ->", d["meta"]["nodes_total"], "nodos")
lv = by_dim(g, "level", "L1-L5")
print("dim level=L1-L5           ->", lv["meta"]["nodes_total"], "nodos (documents con span L1-L5)")

answer       nodes= 7887  edges_inside_view=0
certificate  nodes=   19  edges_inside_view=0
claim        nodes=  544  edges_inside_view=0
decision     nodes=   56  edges_inside_view=0
document     nodes=   13  edges_inside_view=0
experiment   nodes=   14  edges_inside_view=0
gap          nodes=    6  edges_inside_view=0
predicate    nodes=  363  edges_inside_view=404
rule         nodes=   15  edges_inside_view=0

dim truth=Truth.FALSE     -> 7110 nodos, 7059 aristas internas
dim status=DECLARED       -> 521 nodos
dim level=L1-L5           -> 6 nodos (documents con span L1-L5)


## ③ NO-NEW-TRUTH — borrar la cache ≡ reconstruir desde fuentes

La cache (`state/epistemic/graph_cache.json`) es una **materialización
desechable** por fingerprint de contenido de las fuentes. La borramos y
reconstruimos: si byte a byte es idéntico, el grafo no guarda verdad
propia — la verdad vive en los sidecars/ledgers. (Además: tocar un
archivo sin cambiar bytes **no** invalida la cache; cambiar un byte sí.)

In [ ]:
digest = lambda gg: hashlib.sha256(json.dumps(
    {"n": gg["nodes"], "e": gg["edges"]}, sort_keys=True).encode()).hexdigest()
before = digest(g)

from pathlib import Path
cache = Path("state/epistemic/graph_cache.json")
had = cache.exists()
cache.unlink(missing_ok=True)
g2 = build()                      # reconstruccion completa desde fuentes
after = digest(g2)

print("cache existed:", had, "| digest antes:", before[:16])
print("                       | digest despues:", after[:16])
print("RECONSTRUCTION == ORIGINAL :", before == after)
assert before == after, "NO-NEW-TRUTH violado: el grafo guardo verdad propia"
print("cache_hit tras rebuild     :", g2["meta"]["cache_hit"])

cache existed: True | digest antes: 0df292c2a7019a66
                       | digest despues: 0df292c2a7019a66
RECONSTRUCTION == ORIGINAL : True
cache_hit tras rebuild     : False


## ④ Exports — DOT (graphviz), JSON (gephi/d3), networkx (in-process)

Deterministas (R10): nodos ordenados por id, aristas por
`(kind, source, target)`. Cada arista lleva `kind` + `weight`; el estilo
DOT distingue las 6 relaciones ratificadas.

In [ ]:
dot = to_dot(g, name="epistemic_nb08")
print(dot[:520])
print("...")
env = to_json(g)
print("json envelope keys:", sorted(env.keys()))
print("aristas por kind  :", json.dumps(m["edges_by_kind"]))
try:
    import networkx as nx
    gg = __import__("epistemic_graph").to_networkx(g)
    print("networkx         :", type(gg).__name__, gg.number_of_nodes(), "nodos,",
          gg.number_of_edges(), "aristas")
except Exception as e:
    print("networkx no disponible:", e)

digraph epistemic_nb08 {
  rankdir=TB; node [fontname="monospace", fontsize=9];
  n0 [label="trace-1790861064399-132833 → Truth.FALSE\n[answer]", shape=ellipse, style=filled, fillcolor="#f8d7da", tooltip="answer:trace-1790861064399-132833|Truth.FALSE\ncertified=True\nfamily=AND\nsource=state/rsi/tree_traces.jsonl\ntruth=Truth.FALSE\nts=2026-10-01T13:24:24.399056+00:00"];
  n1 [label="trace-1790861090713-132951 → Truth.FALSE\n[answer]", shape=ellipse, style=filled, fillcolor="#f8d7da", tooltip="answer:trace-17908610
...
json envelope keys: ['graph', 'links', 'meta', 'nodes']
aristas por kind  : {"answers": 7887, "certified_by": 19, "child_of": 404, "correlates_with": 30, "derived_from": 19, "extracted_from": 457}
networkx         : DiGraph 8917 nodos, 8816 aristas


## ⑤ Denominadores honestos — lo que el grafo NO pudo conectar

Nada de esto es silencioso: cada skip es un contador en `meta.skipped`
(regla de denominador). Son candidatos de trabajo, no bugs ocultos:

- `claims_without_document` — claims cuyo `subject` no tiene sidecar
  `.depth.vsm`/documento (stem no emparejado).
- `exp_tokens_without_experiment` — tokens `EXP-*` citados en claims que
  no existen como nodo (fuente canónica: `paper_data.vsm`).
- `claim_duplicate_content` — claims con mismo `source_id+text_hash`
  (dedupe content-keyed, D3).
- `decision_duplicate` — un mismo ref aparece en focus **y** registry
  (primera aparición gana, contado).

**Fase-2 (diseñada, no implementada):** aristas `contradicts`,
`refutes`, `supersedes` + vistas cruzadas claim→predicate que lo
certificó, decision→gaps resueltos, experiment→claim correlacionado —
pendiente de aprobación S5 sobre el diseño ya ratificado en D4-backlog.

In [ ]:
summary = {
  "nb": "NB08 — grafo epistémico (D3/D4 ratificados 2026-10-10)",
  "proposal": "docs/plans/graph_epistemic_proposal.vsm @status ratified",
  "mvp": {
    "sources": 7,
    "node_kinds": m["kinds"],
    "edge_kinds": m["edge_kinds"],
    "nodes_total": m["nodes_total"],
    "edges_total": m["edges_total"],
    "all_6_edge_kinds_present": sorted(m["edges_by_kind"]) == sorted(m["edge_kinds"]),
    "all_9_node_kinds_present": sorted(m["nodes_by_kind"]) == sorted(m["kinds"]),
  },
  "checks": {
    "R10_determinism": "assert en tests/test_build.py (mismo input -> mismo grafo)",
    "D3_content_keyed": "identidad sobrevive touch; cambia con bytes del artefacto",
    "NO-NEW-TRUTH": "cache unlink -> rebuild identico (celda ③)",
    "tests": "24/24 pytest packages/framework/epistemic_graph/tests",
  },
  "fase2_backlog": ["contradicts", "refutes", "supersedes"],
}
print(json.dumps(summary, indent=1, ensure_ascii=False))

{
 "nb": "NB08 — grafo epistémico (D3/D4 ratificados 2026-10-10)",
 "proposal": "docs/plans/graph_epistemic_proposal.vsm @status ratified",
 "mvp": {
  "sources": 7,
  "node_kinds": [
   "document",
   "claim",
   "predicate",
   "certificate",
   "answer",
   "experiment",
   "decision",
   "gap",
   "rule"
  ],
  "edge_kinds": [
   "child_of",
   "extracted_from",
   "certified_by",
   "derived_from",
   "correlates_with",
   "answers"
  ],
  "nodes_total": 8917,
  "edges_total": 8816,
  "all_6_edge_kinds_present": true,
  "all_9_node_kinds_present": true
 },
 "checks": {
  "R10_determinism": "assert en tests/test_build.py (mismo input -> mismo grafo)",
  "D3_content_keyed": "identidad sobrevive touch; cambia con bytes del artefacto",
  "NO-NEW-TRUTH": "cache unlink -> rebuild identico (celda ③)",
  "tests": "24/24 pytest packages/framework/epistemic_graph/tests"
 },
 "fase2_backlog": [
  "contradicts",
  "refutes",
  "supersedes"
 ]
}
